# 06 — Control Pair (Peaberry–Longberry)

Control experiment for the choice of the merged pair. The two-stage approach is rebuilt with peaberry and longberry merged at stage 1, a pair that is visually dissimilar according to the class descriptions of the dataset (Febriana et al., 2022). Every training setting, including the unweighted loss and the seeds, is identical to notebook 01. The flat and main two-stage models from notebooks 01–02 are reused. All settings and analyses below were fixed before training, and the results are reported regardless of their direction.

## 1. Configuration

In [1]:
import torch
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# Paths (adjust to your Google Drive)
DATA_DIR  = Path('/content/drive/MyDrive/USK-Coffee')
LOCAL_DIR = Path('/content/USK-Coffee')
OUT_ROOT  = Path('/content/drive/MyDrive/coffee-two-stage/models')               # models from notebook 01
C_ROOT    = Path('/content/drive/MyDrive/coffee-two-stage/models_control_pair')  # new control-pair models
TEST_OUT  = Path('/content/drive/MyDrive/coffee-two-stage/test_results_control_pair')

# Data
CLASSES = ['defect', 'longberry', 'peaberry', 'premium']
EXPECTED_COUNT = {'train': 1200, 'val': 400, 'test': 400}
IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Training (same as notebook 01)
BATCH_SIZE = 32
LR = 1e-4
STEP_SIZE, GAMMA = 5, 0.5
MAX_EPOCHS = 30
PATIENCE = 5
NUM_WORKERS = 2

ARCHS = ['resnet50', 'efficientnet_b0']
SEEDS = [0, 1, 2]
ALPHA = 0.05

# Control pair merged at stage 1
# Stage 1: 0 defect, 1 premium, 2 peaberry+longberry; stage 2: 0 longberry, 1 peaberry
SCHEMES_MAP = {
    '4class':      {'defect': 0, 'longberry': 1, 'peaberry': 2, 'premium': 3},
    '3class_ctrl': {'defect': 0, 'premium': 1, 'peaberry': 2, 'longberry': 2},
    '2class_ctrl': {'longberry': 0, 'peaberry': 1},
}
NUM_CLASSES = {'3class_ctrl': 3, '2class_ctrl': 2}
CTRL_SCHEMES = ['3class_ctrl', '2class_ctrl']

def run_dir(arch, scheme, seed):
    return OUT_ROOT / arch / scheme / f'seed{seed}'

def c_dir(arch, scheme, seed):
    return C_ROOT / arch / scheme / f'seed{seed}'

TEST_OUT.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


## 2. Copy data to the Colab disk

In [2]:
import shutil, time

EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

def list_images(folder):
    return sorted(f for f in folder.iterdir() if f.suffix.lower() in EXTS)

t0 = time.time()
for split in ['train', 'val', 'test']:
    dst = LOCAL_DIR / split
    ok = dst.exists() and all(
        (dst / c).exists() and len(list_images(dst / c)) == EXPECTED_COUNT[split]
        for c in CLASSES)
    if not ok:
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(DATA_DIR / split, dst)
    for c in CLASSES:
        assert len(list_images(dst / c)) == EXPECTED_COUNT[split], f'{split}/{c} incomplete'
print(f'done in {time.time() - t0:.0f}s')

done in 122s


## 3. Dataset and transforms

In [3]:
import random
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Same augmentation as notebook 01
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class CoffeeDataset(Dataset):
    def __init__(self, split_dir, scheme, transform):
        self.samples = [(f, label) for c, label in SCHEMES_MAP[scheme].items()
                        for f in list_images(split_dir / c)]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        with Image.open(path) as img:
            return self.transform(img.convert('RGB')), label

## 4. Training function

In [4]:
import pandas as pd
import torch.nn as nn
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(_):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

def build_model(arch, num_classes, pretrained=True):
    if arch == 'resnet50':
        m = models.resnet50(weights='IMAGENET1K_V1' if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif arch == 'efficientnet_b0':
        m = models.efficientnet_b0(weights='IMAGENET1K_V1' if pretrained else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    else:
        raise ValueError(arch)
    return m

@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, labels = [], []
    for x, y in loader:
        probs.append(torch.softmax(model(x.to(device)), 1).cpu())
        labels.append(y)
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()

def epochs_since_improvement(history):
    losses = [h['val_loss'] for h in history]
    return len(losses) - 1 - int(np.argmin(losses))

def should_stop(history):
    return len(history) >= MAX_EPOCHS or (len(history) > 0 and epochs_since_improvement(history) >= PATIENCE)

# Identical to train_model in notebook 01 (unweighted loss); only the label scheme differs
def train_model(arch, scheme, seed, out_dir, pretrained=True):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path  = out_dir / 'last_checkpoint.pt'
    best_path  = out_dir / 'best_model.pt'
    final_path = out_dir / 'final_model.pt'
    if final_path.exists():
        print(f'skip (done): {out_dir}')
        return

    set_seed(seed)
    train_ds = CoffeeDataset(LOCAL_DIR / 'train', scheme, train_tf)
    val_ds   = CoffeeDataset(LOCAL_DIR / 'val', scheme, eval_tf)
    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                              worker_init_fn=seed_worker, generator=g, pin_memory=True)
    val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model = build_model(arch, NUM_CLASSES[scheme], pretrained).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)

    history = []
    # Resume after a Colab disconnect
    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer'])
        scheduler.load_state_dict(ck['scheduler']); history = ck['history']
        g.set_state(ck['rng_gen']); torch.set_rng_state(ck['rng_torch'])
        np.random.set_state(ck['rng_np']); random.setstate(ck['rng_py'])
        if ck['rng_cuda'] is not None: torch.cuda.set_rng_state_all(ck['rng_cuda'])
        print(f'resuming from epoch {len(history) + 1}')

    while not should_stop(history):
        epoch = len(history) + 1
        t0 = time.time()
        lr = optimizer.param_groups[0]['lr']
        model.train()
        loss_sum, correct, total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item(); total += len(y)
        scheduler.step()

        prob, y_val = predict(model, val_loader)
        val_loss = nn.functional.nll_loss(torch.log(torch.tensor(prob) + 1e-12), torch.tensor(y_val)).item()
        row = {'epoch': epoch, 'lr': lr, 'train_loss': loss_sum / total, 'train_acc': correct / total,
               'val_loss': val_loss, 'val_acc': float((prob.argmax(1) == y_val).mean()), 'sec': time.time() - t0}

        # Keep the weights with the lowest validation loss
        if row['val_loss'] < min((h['val_loss'] for h in history), default=float('inf')):
            torch.save(model.state_dict(), best_path)
        history.append(row)

        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(), 'history': history,
                    'rng_gen': g.get_state(), 'rng_torch': torch.get_rng_state(),
                    'rng_np': np.random.get_state(), 'rng_py': random.getstate(),
                    'rng_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None},
                   ckpt_path)
        pd.DataFrame(history).to_csv(out_dir / 'history.csv', index=False)
        print(f"  epoch {epoch:>2}/{MAX_EPOCHS} | train {row['train_loss']:.4f}/{row['train_acc']:.4f} "
              f"| val {val_loss:.4f}/{row['val_acc']:.4f} | {row['sec']:.0f}s")

    hist = pd.DataFrame(history)
    best = hist.loc[hist['val_loss'].idxmin()]
    print(f"  stopped after epoch {len(hist)} | best val_loss {best['val_loss']:.4f} "
          f"(val_acc {best['val_acc']:.4f}) at epoch {int(best['epoch'])}")

    model.load_state_dict(torch.load(best_path, map_location=device))
    full_val = CoffeeDataset(LOCAL_DIR / 'val', '4class', eval_tf)
    prob, y_true = predict(model, DataLoader(full_val, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS))
    np.savez(out_dir / 'val_predictions.npz', prob=prob, y_true=y_true,
             paths=np.array([str(p.relative_to(LOCAL_DIR)) for p, _ in full_val.samples]))

    best_path.replace(final_path)
    ckpt_path.unlink(missing_ok=True)

## 5. Train the 12 control models

In [5]:
# Completed runs are skipped, interrupted runs resume
runs = [(seed, arch, scheme) for seed in SEEDS for arch in ARCHS for scheme in CTRL_SCHEMES]

for i, (seed, arch, scheme) in enumerate(runs, 1):
    print(f'[{i:>2}/{len(runs)}] {arch} | {scheme} | seed {seed}')
    train_model(arch, scheme, seed, c_dir(arch, scheme, seed))
print('All runs finished')

[ 1/12] resnet50 | 3class_ctrl | seed 0
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 119MB/s]


  epoch  1/30 | train 0.4817/0.8004 | val 0.4741/0.8500 | 66s
  epoch  2/30 | train 0.2542/0.9077 | val 0.3058/0.8894 | 65s
  epoch  3/30 | train 0.1859/0.9337 | val 1.0159/0.7175 | 73s
  epoch  4/30 | train 0.1454/0.9506 | val 0.3055/0.8869 | 72s
  epoch  5/30 | train 0.1150/0.9594 | val 0.2995/0.8912 | 75s
  epoch  6/30 | train 0.0680/0.9758 | val 0.1496/0.9456 | 75s
  epoch  7/30 | train 0.0483/0.9844 | val 0.1616/0.9494 | 75s
  epoch  8/30 | train 0.0528/0.9808 | val 0.2450/0.9137 | 75s
  epoch  9/30 | train 0.0485/0.9848 | val 0.2332/0.9363 | 74s
  epoch 10/30 | train 0.0490/0.9817 | val 0.1665/0.9437 | 73s
  epoch 11/30 | train 0.0410/0.9858 | val 0.1727/0.9475 | 71s
  stopped after epoch 11 | best val_loss 0.1496 (val_acc 0.9456) at epoch 6
[ 2/12] resnet50 | 2class_ctrl | seed 0
  epoch  1/30 | train 0.1601/0.9346 | val 0.0705/0.9738 | 39s
  epoch  2/30 | train 0.0464/0.9858 | val 0.0872/0.9725 | 35s
  epoch  3/30 | train 0.0312/0.9908 | val 0.0820/0.9738 | 37s
  epoch  4/30 | 

100%|██████████| 20.5M/20.5M [00:00<00:00, 158MB/s]


  epoch  1/30 | train 0.6919/0.7017 | val 0.4890/0.8131 | 59s
  epoch  2/30 | train 0.3100/0.8848 | val 0.3842/0.8575 | 54s
  epoch  3/30 | train 0.2222/0.9131 | val 0.3997/0.8588 | 55s
  epoch  4/30 | train 0.1611/0.9415 | val 0.2938/0.8956 | 53s
  epoch  5/30 | train 0.1435/0.9502 | val 0.2117/0.9269 | 54s
  epoch  6/30 | train 0.1110/0.9613 | val 0.1931/0.9356 | 52s
  epoch  7/30 | train 0.1076/0.9633 | val 0.2392/0.9306 | 54s
  epoch  8/30 | train 0.0900/0.9690 | val 0.2335/0.9281 | 52s
  epoch  9/30 | train 0.0753/0.9750 | val 0.1964/0.9331 | 57s
  epoch 10/30 | train 0.0707/0.9752 | val 0.1857/0.9469 | 57s
  epoch 11/30 | train 0.0717/0.9756 | val 0.1677/0.9450 | 56s
  epoch 12/30 | train 0.0642/0.9767 | val 0.1759/0.9419 | 57s
  epoch 13/30 | train 0.0554/0.9804 | val 0.1641/0.9481 | 56s
  epoch 14/30 | train 0.0589/0.9785 | val 0.1784/0.9450 | 57s
  epoch 15/30 | train 0.0484/0.9844 | val 0.1758/0.9425 | 54s
  epoch 16/30 | train 0.0467/0.9850 | val 0.1552/0.9506 | 56s
  epoch 

## 6. Training status

In [6]:
status = []
for seed, arch, scheme in runs:
    d = c_dir(arch, scheme, seed)
    row = {'arch': arch, 'scheme': scheme, 'seed': seed}
    if (d / 'history.csv').exists():
        h = pd.read_csv(d / 'history.csv')
        b = h.loc[h['val_loss'].idxmin()]
        row.update({'epochs_run': len(h), 'best_epoch': int(b['epoch']),
                    'best_val_loss': round(b['val_loss'], 4), 'val_acc_at_best': round(b['val_acc'], 4),
                    'train_minutes': round(h['sec'].sum() / 60, 1)})
    row['status'] = ('done' if (d / 'final_model.pt').exists()
                     else 'running/interrupted' if (d / 'history.csv').exists() else 'not started')
    status.append(row)
status = pd.DataFrame(status)
status.to_csv(TEST_OUT / 'training_status_control.csv', index=False)
print(status.to_string(index=False))

           arch      scheme  seed  epochs_run  best_epoch  best_val_loss  val_acc_at_best  train_minutes status
       resnet50 3class_ctrl     0          11           6         0.1496           0.9456           13.2   done
       resnet50 2class_ctrl     0          20          15         0.0207           0.9938           13.0   done
efficientnet_b0 3class_ctrl     0          22          17         0.1525           0.9519           20.2   done
efficientnet_b0 2class_ctrl     0           8           3         0.0089           0.9988            3.8   done
       resnet50 3class_ctrl     1          10           5         0.1643           0.9469           12.2   done
       resnet50 2class_ctrl     1           8           3         0.0421           0.9875            5.3   done
efficientnet_b0 3class_ctrl     1          15          10         0.1568           0.9450           14.1   done
efficientnet_b0 2class_ctrl     1           8           3         0.0159           0.9950            3.8

## 7. Test predictions

In [7]:
test_ds = CoffeeDataset(LOCAL_DIR / 'test', '4class', eval_tf)
test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_paths = np.array([str(p.relative_to(LOCAL_DIR)) for p, _ in test_ds.samples])

for seed, arch, scheme in runs:
    d = c_dir(arch, scheme, seed)
    f = d / 'test_predictions.npz'
    if f.exists():
        print(f'skip (already predicted): {d}')
        continue
    model = build_model(arch, NUM_CLASSES[scheme], pretrained=False)
    model.load_state_dict(torch.load(d / 'final_model.pt', map_location='cpu'))
    prob, y_true = predict(model.to(device), test_loader)
    np.savez(f, prob=prob, y_true=y_true, paths=test_paths)
    print(f'predicted: {d}')

## 8. Metrics per seed

In [8]:
from sklearn.metrics import precision_recall_fscore_support

def load_npz(path):
    z = np.load(path)
    assert (z['paths'] == test_paths).all(), f'image order mismatch: {path}'
    return z

# Main two-stage: stage 1 = 0 defect, 1 longberry, 2 peaberry+premium; stage 2 = 0 peaberry, 1 premium
def main_predict(s1_prob, s2_prob):
    s1, s2 = s1_prob.argmax(1), s2_prob.argmax(1)
    return np.where(s1 == 2, 2 + s2, s1)

# Control two-stage: stage 1 = 0 defect, 1 premium, 2 peaberry+longberry; stage 2 = 0 longberry, 1 peaberry
def control_predict(s1_prob, s2_prob):
    s1, s2 = s1_prob.argmax(1), s2_prob.argmax(1)
    merged = np.where(s2 == 0, 1, 2)
    return np.select([s1 == 0, s1 == 1], [0, 3], default=merged)

def pair_errors(y, pred, a, b):
    return int(((y == a) & (pred == b)).sum() + ((y == b) & (pred == a)).sum())

def metrics_row(y, pred):
    p, r, f1, _ = precision_recall_fscore_support(y, pred, labels=range(4), zero_division=0)
    errors = int((y != pred).sum())
    row = {'accuracy': (y == pred).mean(), 'macro_f1': f1.mean(), 'errors': errors,
           'pb_pr_errors': pair_errors(y, pred, 2, 3),   # main pair
           'pb_lb_errors': pair_errors(y, pred, 2, 1)}   # control pair
    for k, c in enumerate(CLASSES):
        row.update({f'P_{c}': p[k], f'R_{c}': r[k], f'F1_{c}': f1[k]})
    return row

def control_stage_details(y, s1_prob, s2_prob):
    s1_pred = s1_prob.argmax(1)
    pair = np.isin(y, [1, 2])
    s1_true = np.select([y == 0, y == 3], [0, 1], default=2)
    s2_true = (y == 2).astype(int)
    return {'stage1_acc': (s1_true == s1_pred).mean(),
            'stage2_acc_on_pair': (s2_prob.argmax(1)[pair] == s2_true[pair]).mean(),
            'pair_lost_at_stage1': int((pair & (s1_pred != 2)).sum()),
            'other_routed_to_stage2': int((~pair & (s1_pred == 2)).sum()),
            'routed_to_stage2_share': (s1_pred == 2).mean()}

rows, preds = [], {}
for arch in ARCHS:
    for seed in SEEDS:
        flat = load_npz(run_dir(arch, '4class', seed) / 'test_predictions.npz')
        s1   = load_npz(run_dir(arch, '3class', seed) / 'test_predictions.npz')
        s2   = load_npz(run_dir(arch, '2class', seed) / 'test_predictions.npz')
        c1   = load_npz(c_dir(arch, '3class_ctrl', seed) / 'test_predictions.npz')
        c2   = load_npz(c_dir(arch, '2class_ctrl', seed) / 'test_predictions.npz')
        y = flat['y_true']
        pred_flat = flat['prob'].argmax(1)
        pred_main = main_predict(s1['prob'], s2['prob'])
        pred_ctrl = control_predict(c1['prob'], c2['prob'])
        preds[(arch, seed)] = (y, pred_flat, pred_main, pred_ctrl)

        rows.append({'arch': arch, 'seed': seed, 'model': 'flat', **metrics_row(y, pred_flat)})
        rows.append({'arch': arch, 'seed': seed, 'model': 'hierarchical', **metrics_row(y, pred_main)})
        rows.append({'arch': arch, 'seed': seed, 'model': 'hierarchical_control',
                     **metrics_row(y, pred_ctrl), **control_stage_details(y, c1['prob'], c2['prob'])})

results = pd.DataFrame(rows)
results.to_csv(TEST_OUT / 'test_results_per_seed_control.csv', index=False)

main_cols = ['accuracy', 'macro_f1', 'pb_pr_errors', 'pb_lb_errors', 'errors']
stage_cols = ['stage1_acc', 'stage2_acc_on_pair', 'pair_lost_at_stage1',
              'other_routed_to_stage2', 'routed_to_stage2_share']
pd.set_option('display.width', 220)
print(results[['arch', 'model', 'seed'] + main_cols].round(4).to_string(index=False))
print('\nControl stage details:')
print(results[results['model'] == 'hierarchical_control'][['arch', 'seed'] + stage_cols].round(4).to_string(index=False))

           arch                model  seed  accuracy  macro_f1  pb_pr_errors  pb_lb_errors  errors
       resnet50                 flat     0    0.8019    0.8002           112             9     317
       resnet50         hierarchical     0    0.8162    0.8201            83            20     294
       resnet50 hierarchical_control     0    0.8256    0.8292           108            16     279
       resnet50                 flat     1    0.7950    0.8002            93            29     328
       resnet50         hierarchical     1    0.8662    0.8665            41            22     214
       resnet50 hierarchical_control     1    0.8506    0.8501            69            28     239
       resnet50                 flat     2    0.8725    0.8735            64             4     204
       resnet50         hierarchical     2    0.8281    0.8306            62            21     275
       resnet50 hierarchical_control     2    0.8406    0.8416            66             4     255
efficientn

## 9. Mean and standard deviation

In [9]:
summary = results.groupby(['arch', 'model'], sort=False)[main_cols + stage_cols].agg(['mean', 'std'])
summary.to_csv(TEST_OUT / 'test_results_summary_control.csv')
print(summary.round(4).to_string())

                                     accuracy         macro_f1         pb_pr_errors          pb_lb_errors             errors          stage1_acc         stage2_acc_on_pair         pair_lost_at_stage1          other_routed_to_stage2          routed_to_stage2_share        
                                         mean     std     mean     std         mean      std         mean      std      mean      std       mean     std               mean     std                mean      std                   mean      std                   mean     std
arch            model                                                                                                                                                                                                                                                          
resnet50        flat                   0.8231  0.0429   0.8246  0.0423      89.6667  24.1730      14.0000  13.2288  283.0000  68.6367        NaN     NaN                NaN     NaN     

## 10. McNemar test (control two-stage vs flat)

In [10]:
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

# Same procedure as notebook 02; the second subset is the control pair (longberry + peaberry)
tests = []
for arch in ARCHS:
    for seed in SEEDS:
        y, pf, _, pc = preds[(arch, seed)]
        for subset, mask in [('all_1600', np.ones_like(y, bool)), ('pb_lb_800', np.isin(y, [1, 2]))]:
            cf, cc = (pf == y)[mask], (pc == y)[mask]
            b = int((cf & ~cc).sum())   # correct by flat only
            c = int((~cf & cc).sum())   # correct by control two-stage only
            table = [[int((cf & cc).sum()), b], [c, int((~cf & ~cc).sum())]]
            direction = 'ctrl_better' if c > b else ('flat_better' if b > c else 'equal')
            tests.append({'arch': arch, 'seed': seed, 'subset': subset,
                          'acc_flat': cf.mean(), 'acc_ctrl': cc.mean(),
                          'b_flat_only': b, 'c_ctrl_only': c, 'direction': direction,
                          'p_exact': mcnemar(table, exact=True).pvalue})

mc = pd.DataFrame(tests)
mc['p_holm'] = np.nan
for arch in ARCHS:
    idx = mc['arch'] == arch
    mc.loc[idx, 'p_holm'] = multipletests(mc.loc[idx, 'p_exact'], alpha=ALPHA, method='holm')[1]
mc['significant'] = mc['p_holm'] < ALPHA
mc.to_csv(TEST_OUT / 'mcnemar_results_control.csv', index=False)
print(mc.round(4).to_string(index=False))

# Consistency criterion: significant in all three seeds in the same direction
rows = []
for (arch, subset), g in mc.groupby(['arch', 'subset'], sort=False):
    n_ctrl = int((g['significant'] & (g['direction'] == 'ctrl_better')).sum())
    n_flat = int((g['significant'] & (g['direction'] == 'flat_better')).sum())
    if n_ctrl == len(SEEDS):
        verdict = 'consistent: control two-stage better'
    elif n_flat == len(SEEDS):
        verdict = 'consistent: flat better'
    elif n_ctrl + n_flat > 0:
        verdict = 'inconsistent across seeds'
    else:
        verdict = 'no significant difference'
    rows.append({'arch': arch, 'subset': subset, 'sig_ctrl_better': f'{n_ctrl}/{len(SEEDS)}',
                 'sig_flat_better': f'{n_flat}/{len(SEEDS)}', 'verdict': verdict})
conclusion = pd.DataFrame(rows)
conclusion.to_csv(TEST_OUT / 'mcnemar_conclusion_control.csv', index=False)
print(conclusion.to_string(index=False))

           arch  seed    subset  acc_flat  acc_ctrl  b_flat_only  c_ctrl_only   direction  p_exact  p_holm  significant
       resnet50     0  all_1600    0.8019    0.8256           92          130 ctrl_better   0.0128  0.0257         True
       resnet50     0 pb_lb_800    0.8238    0.8025           74           57 flat_better   0.1619  0.1619        False
       resnet50     1  all_1600    0.7950    0.8506           63          152 ctrl_better   0.0000  0.0000         True
       resnet50     1 pb_lb_800    0.7725    0.9025           12          116 ctrl_better   0.0000  0.0000         True
       resnet50     2  all_1600    0.8725    0.8406          111           60 flat_better   0.0001  0.0005         True
       resnet50     2 pb_lb_800    0.9062    0.8625           58           23 flat_better   0.0001  0.0005         True
efficientnet_b0     0  all_1600    0.8250    0.8119           95           74 flat_better   0.1237  0.3710        False
efficientnet_b0     0 pb_lb_800    0.798

## 11. Mean difference across seeds with 95% CI

In [11]:
from scipy import stats

# Same as the manuscript: Student's t, df = number of seeds - 1
t_crit = stats.t.ppf(0.975, len(SEEDS) - 1)
ci_rows = []
for arch in ARCHS:
    for model_name, subset, cls in [('hierarchical', 'all_1600', None),
                                    ('hierarchical_control', 'all_1600', None),
                                    ('hierarchical', 'pb_pr_800', [2, 3]),
                                    ('hierarchical_control', 'pb_lb_800', [1, 2])]:
        diffs = []
        for seed in SEEDS:
            y, pf, pm, pc = preds[(arch, seed)]
            mask = np.ones_like(y, bool) if cls is None else np.isin(y, cls)
            p_model = pm if model_name == 'hierarchical' else pc
            diffs.append(100 * ((p_model == y)[mask].mean() - (pf == y)[mask].mean()))
        diffs = np.array(diffs)
        half = t_crit * diffs.std(ddof=1) / np.sqrt(len(diffs))
        ci_rows.append({'arch': arch, 'model': model_name, 'subset': subset,
                        **{f'diff_seed{s}': round(d, 2) for s, d in zip(SEEDS, diffs)},
                        'mean_diff': round(diffs.mean(), 2),
                        'ci_low': round(diffs.mean() - half, 2), 'ci_high': round(diffs.mean() + half, 2)})
ci = pd.DataFrame(ci_rows)
ci.to_csv(TEST_OUT / 'ci_difference_control.csv', index=False)
print(ci.to_string(index=False))

           arch                model    subset  diff_seed0  diff_seed1  diff_seed2  mean_diff  ci_low  ci_high
       resnet50         hierarchical  all_1600        1.44        7.12       -4.44       1.37  -12.99    15.74
       resnet50 hierarchical_control  all_1600        2.38        5.56       -3.19       1.58   -9.42    12.58
       resnet50         hierarchical pb_pr_800        3.25        4.87       -0.50       2.54   -4.31     9.39
       resnet50 hierarchical_control pb_lb_800       -2.12       13.00       -4.37       2.17  -21.31    25.64
efficientnet_b0         hierarchical  all_1600        1.69        2.31       -2.31       0.56   -5.67     6.80
efficientnet_b0 hierarchical_control  all_1600       -1.31       -1.81       -4.69      -2.60   -7.13     1.92
efficientnet_b0         hierarchical pb_pr_800        4.25        2.25        0.13       2.21   -2.92     7.33
efficientnet_b0 hierarchical_control pb_lb_800        0.50       -1.75       -3.37      -1.54   -6.38     3.29
